In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FFMpegWriter  # ou PillowWriter
import os
import json
import glob
from matplotlib import gridspec
import imageio.v2 as imageio
import matplotlib as mpl
from PIL import Image, ImageDraw, ImageFont
from src.network_animation import convert_positions_3D
from mayavi import mlab
import pandas as pd

## GROWTH NETWORK - 3D

In [7]:
dim = 3
L = 256
nc = 4
rho = 1/nc
k = 1.0e-06
Nt = 3000


# pasta onde estão os frames
frames_dir  = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/positions_vs_time"
# padrão dos arquivos
pattern = os.path.join(frames_dir, "frame_*.png")

# lista de frames ordenada
frame_files = sorted(glob.glob(pattern))

print(f"Encontrados {len(frame_files)} frames.")
if not frame_files:
    raise RuntimeError("Nenhum frame_*.png encontrado!")

frames_out = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/"
# nome do gif de saída
out_gif = os.path.join(frames_out, "positions_3D_anim.gif")

# fps desejado
fps = 20
frame_duration = 1.0 / fps  # segundos por frame

# cria o GIF em streaming (sem carregar tudo em memória)
with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i, fname in enumerate(frame_files):
        img = imageio.imread(fname)
        writer.append_data(img)

        if (i % 50) == 0 or i == len(frame_files) - 1:
            print(f"[{i+1:5d}/{len(frame_files)}]  adicionado: {os.path.basename(fname)}")

print("GIF salvo em:", out_gif)


Encontrados 578 frames.
[    1/578]  adicionado: frame_0000.png
[   51/578]  adicionado: frame_0050.png
[  101/578]  adicionado: frame_0100.png
[  151/578]  adicionado: frame_0150.png
[  201/578]  adicionado: frame_0200.png
[  251/578]  adicionado: frame_0250.png
[  301/578]  adicionado: frame_0300.png
[  351/578]  adicionado: frame_0350.png
[  401/578]  adicionado: frame_0400.png
[  451/578]  adicionado: frame_0450.png
[  501/578]  adicionado: frame_0500.png
[  551/578]  adicionado: frame_0550.png
[  578/578]  adicionado: frame_0577.png
GIF salvo em: ../animation/3D_L256_nc4_rho0.25_k1.0e-06_Nt3000/positions_3D_anim.gif


## GROWTH PROPERTIES

In [4]:
plt.style.use('properties.mplstyle')

In [8]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import imageio.v2 as imageio
import matplotlib as mpl

# ---------------- Parâmetros base ----------------
dim = 3
L   = 256          # usado no path; L real vem do shape no npz
nc  = 4
rho = 1/nc
k   = 1.0e-06
Nt  = 3000

# ---------------- Paths ----------------
path_json = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.json"
path_npz  = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.npz"

# mesmas cores do crescimento da rede
colors_used = [
    (0.9, 0.1, 0.1),  # Color 1 - red
    (0.1, 0.9, 0.1),  # Color 2 - green
    (0.1, 0.1, 0.9),  # Color 3 - blue
    (0.2, 0.8, 0.8),  # Color 4 - teal
]

out_dir = os.path.dirname(path_json)
out_gif = os.path.join(out_dir, "positions_properties.gif")

# ============================================================
# 1) Carrega o NPZ para pegar frame_times (mesmos da rede)
# ============================================================
data_npz = np.load(path_npz)
shape      = tuple(data_npz["shape"])
M_flat     = data_npz["data"]
num_colors = int(data_npz["num_colors"])

print("NPZ shape salvo:", shape)
print("num_colors:", num_colors)

assert np.prod(shape) == M_flat.size, "shape x size inconsistente"

M = M_flat.reshape(shape)
print("M.shape real:", M.shape)

# Decodificação M = c*1e6 + t
M = M.astype(np.int32, copy=False)
color_mat = (M // 100_000_000).astype(np.int16)   # 1..num_colors
time_raw  = (M %  100_000_000).astype(np.int32)   # 0..tmax ou 999999

print("cores não nulas (NPZ):", np.unique(color_mat[color_mat > 0]))

T_SENTINELA = 999_999
mask_valid_time = (time_raw < T_SENTINELA) & (color_mat > 0)
print("N sítios com tempo válido:", mask_valid_time.sum())

t_vals = np.unique(time_raw[mask_valid_time])
t_min_phys, t_max_phys = int(t_vals[0]), int(t_vals[-1])
print(f"t físico min: {t_min_phys}, max: {t_max_phys}, qtd t distintos: {len(t_vals)}")

# Lista de tempos (frames) — deve ser a MESMA usada no GIF da rede
frame_stride = 1        # se mudar aqui, mude também no gif da rede
frame_times  = t_vals[::frame_stride]
n_frames     = frame_times.size
print("Número de frames:", n_frames)

# ============================================================
# 2) Carrega JSON e prepara curvas p(t), N(t)
# ============================================================
with open(path_json, "r") as f:
    data = json.load(f)

results = data["results"]

curves = {}
for v in results.values():
    d = v["data"]
    c = d["color"]
    # guarda qualquer cor definida no JSON
    curves[c] = {
        "time": np.asarray(d["time"], dtype=float),
        "pt":   np.asarray(d["pt"],   dtype=float),
        "nt":   np.asarray(d["nt"],   dtype=float),
    }

colors_available = sorted(curves.keys())
print("cores presentes no JSON:", colors_available)

# assume que todas as cores têm o mesmo vetor de tempo
first_c = colors_available[0]
t_arr = curves[first_c]["time"]  # vetor de tempo do JSON (0..tmax)
t_min, t_max = t_arr[0], t_arr[-1]

# limites globais em Y (pt e Nt) usando TODAS as cores
pt_min = min(curves[c]["pt"].min() for c in colors_available)
pt_max = max(curves[c]["pt"].max() for c in colors_available)

nt_min = min(curves[c]["nt"].min() for c in colors_available)
nt_max = max(curves[c]["nt"].max() for c in colors_available)

pt_margin = 0.05 * (pt_max - pt_min)
nt_margin = 0.05 * (nt_max - nt_min)

thickness_axes = 1.2
fs_ticks = 12

# ============================================================
# 3) Cria figura base
# ============================================================
plt.close("all")
fig = plt.figure(figsize=(6, 6), dpi=150)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.15)

ax_pt = fig.add_subplot(gs[0])
ax_nt = fig.add_subplot(gs[1], sharex=ax_pt)

ax_pt.set_ylabel(r"$p(t)$", fontsize=15)
ax_nt.set_ylabel(r"$N(t)$", fontsize=15)
ax_nt.set_xlabel("$t$", fontsize=15)

# limites em x: tempos físicos da rede (NPZ)
ax_pt.set_xlim(t_min_phys, t_max_phys)
ax_nt.set_xlim(t_min_phys, t_max_phys)

# limites em y: globais entre as cores
ax_pt.set_ylim(pt_min - pt_margin, pt_max + pt_margin)
ax_nt.set_ylim(nt_min - nt_margin, nt_max + nt_margin)

ax_pt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)
ax_nt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)

# esconde labels de x no eixo superior
ax_pt.tick_params(axis='x', which='both', labelbottom=False)

pc_reff = 0.24881182
ax_pt.axhline(y=pc_reff, xmin=0, xmax=1, ls='--', color='k',
              label=f"$p_c = {pc_reff}$")
ax_nt.axhline(y=Nt, xmin=0, xmax=1, ls='--', color='k',
              label=f"$N_T = {Nt}$")

# linhas para TODAS as cores disponíveis
lines_pt = {}
lines_nt = {}
for c in colors_available:
    # c = 1..num_colors; índice da lista é c-1
    col = colors_used[c - 1]
    (lp,) = ax_pt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    (ln,) = ax_nt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    lines_pt[c] = lp
    lines_nt[c] = ln

ax_pt.legend(loc="upper right", fontsize=10)
ax_nt.legend(loc="upper right", fontsize=10)

# aumenta margem esquerda para não cortar N(t)
fig.subplots_adjust(left=0.20, right=0.98, bottom=0.12, top=0.98, hspace=0.10)

# ============================================================
# 4) Gera GIF (sem loop infinito) sincronizado com frame_times
# ============================================================

# frame_duration compatível com fps=20 (como no PillowWriter): 1/20 = 0.05 s
frame_duration = 1.0 / 20.0   # ~20 fps

with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i, t in enumerate(frame_times):
        # máscara de pontos com tempo <= t (usando tempo do JSON)
        mask_t = t_arr <= t

        for c in colors_available:
            lines_pt[c].set_data(t_arr[mask_t], curves[c]["pt"][mask_t])
            lines_nt[c].set_data(t_arr[mask_t], curves[c]["nt"][mask_t])

        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        writer.append_data(frame)

        if (i % 50) == 0 or i == len(frame_times) - 1:
            print(f"[{i+1:5d}/{len(frame_times)}]  t = {t:4d}/{t_max_phys}")

plt.close(fig)
print("GIF salvo em:", out_gif)


NPZ shape salvo: (256, 256, 256)
num_colors: 4
M.shape real: (256, 256, 256)
cores não nulas (NPZ): [1 2 3 4]
N sítios com tempo válido: 9731810
t físico min: 0, max: 577, qtd t distintos: 578
Número de frames: 578
cores presentes no JSON: [1, 2, 3, 4]
[    1/578]  t =    0/577
[   51/578]  t =   50/577
[  101/578]  t =  100/577
[  151/578]  t =  150/577
[  201/578]  t =  200/577
[  251/578]  t =  250/577
[  301/578]  t =  300/577
[  351/578]  t =  350/577
[  401/578]  t =  400/577
[  451/578]  t =  450/577
[  501/578]  t =  500/577
[  551/578]  t =  550/577
[  578/578]  t =  577/577
GIF salvo em: ../animation/3D_L256_nc4_rho0.25_k1.0e-06_Nt3000/positions_properties.gif


## COMBINE GIFS

In [11]:
import os
import numpy as np
import imageio.v2 as imageio
from PIL import Image
import matplotlib.pyplot as plt

# ---------------- Parâmetros base ----------------
dim = 3
L   = 256          # usado no path; L real vem do shape no npz
nc  = 4
rho = 1/nc
k   = 1.0e-06
Nt  = 3000

# Diretório onde estão os GIFs
base_dir   = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/"

left_gif   = os.path.join(base_dir, "positions_3D_anim.gif")
right_gif  = os.path.join(base_dir, "positions_properties.gif")
out_gif    = os.path.join(base_dir, "combined.gif")

# Título em "LaTeX" (mathtext do Matplotlib)
title_text = (
    r"Network with $L = %d$, $n_c = %d$, $\rho = %.2f$, "
    r"$k = %.1e$, $N_T = %d$"
) % (L, nc, rho, k, Nt)

# --------------------------------------------------
# 1) Abre os readers em modo streaming
# --------------------------------------------------
reader_left  = imageio.get_reader(left_gif, mode="I")
reader_right = imageio.get_reader(right_gif, mode="I")

n_left  = reader_left.get_length()
n_right = reader_right.get_length()
n_frames = min(n_left, n_right)

print(f"Frames left: {n_left}, right: {n_right}, usados: {n_frames}")

# --------------------------------------------------
# 2) Descobre tamanhos
# --------------------------------------------------
first_left  = reader_left.get_data(0)
first_right = reader_right.get_data(0)

imL0 = Image.fromarray(first_left)
imR0 = Image.fromarray(first_right)

wL, hL = imL0.size
wR, hR = imR0.size

# Redimensiona o da direita para ter a mesma altura do da esquerda
target_h = hL
scale_R  = target_h / hR
new_wR   = int(wR * scale_R)

combined_w = wL + new_wR

# --------------------------------------------------
# 3) Gera uma faixa de título com Matplotlib (reutilizada em todos os frames)
# --------------------------------------------------
def make_title_strip(width_px, height_px, text):
    dpi = 100
    fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    # fontsize maior para destacar bem
    ax.text(
        0.5, 0.5, text,
        ha="center", va="center",
        fontsize=30
    )

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape((h, w, 3))
    plt.close(fig)
    return Image.fromarray(img)

# altura aproximada do título
title_nominal_h = 80
title_img = make_title_strip(combined_w, title_nominal_h, title_text)
title_h = title_img.height

combined_h = title_h + target_h

# --------------------------------------------------
# 4) Cria GIF combinado em streaming
# --------------------------------------------------

# Mesmo "fps" do PillowWriter da rede (20 fps => 0.05 s por frame)
frame_duration = 1.0 / 20.0

with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i in range(n_frames):
        # Lê frame i de cada GIF
        frameL = reader_left.get_data(i)
        frameR = reader_right.get_data(i)

        imL = Image.fromarray(frameL)
        imR = Image.fromarray(frameR)

        # Redimensiona direita
        imR = imR.resize((new_wR, target_h), resample=Image.BILINEAR)

        # Canvas combinado (fundo branco)
        combined = Image.new("RGB", (combined_w, combined_h), color=(255, 255, 255))

        # Cola o título (já com LaTeX renderizado pelo Matplotlib)
        combined.paste(title_img, (0, 0))

        # Cola os dois GIFs abaixo do título
        combined.paste(imL, (0,      title_h))
        combined.paste(imR, (wL,     title_h))

        # Adiciona frame ao GIF final
        writer.append_data(np.array(combined))

        if (i % 50) == 0 or i == n_frames - 1:
            print(f"Frame {i+1}/{n_frames}")

# Fecha os readers
reader_left.close()
reader_right.close()

print("GIF combinado salvo em:", out_gif)

Frames left: 578, right: 577, usados: 577
Frame 1/577
Frame 51/577
Frame 101/577
Frame 151/577
Frame 201/577
Frame 251/577
Frame 301/577
Frame 351/577
Frame 401/577
Frame 451/577
Frame 501/577
Frame 551/577
Frame 577/577
GIF combinado salvo em: ../animation/3D_L256_nc4_rho0.25_k1.0e-06_Nt3000/combined.gif
